Who talks the most in Terence? With the Perseus TEI editions of all six plays in hand—each marked up with `<sp>` (speech) blocks naming a `<speaker>` and wrapping their `<l>` lines—the question becomes a few `xpath` calls and a `defaultdict`.

This notebook pulls the six plays directly from [`PerseusDL/canonical-latinLit`](https://github.com/PerseusDL/canonical-latinLit) on GitHub, builds a `{speaker: [lines]}` dict per play, and reports both line counts and rough word counts. One caveat worth stating up front: the speaker codes in the source XML are not always normalized—the same character can appear under several abbreviations (`Pa.` / `Pa` / `Paw.` in *Hecyra*, `MIy.` / `My.` in the same play, etc.) and a few stray OCR-style variants slip through. The counts below treat each speaker code as distinct; collapsing them is left as a follow-up exercise.

In [ ]:
# Imports
import urllib.request
from collections import defaultdict
from pprint import pprint

from lxml import etree

In [ ]:
# Perseus URIs for the six Terence plays
# phi0134 = Terence; phi001..phi006 = Andria, Heaut., Eunuchus, Phormio, Hecyra, Adelphoe
play_base = (
    'https://raw.githubusercontent.com/PerseusDL/canonical-latinLit/master/'
    'data/phi0134/'
)
play_ids = ['phi001', 'phi002', 'phi003', 'phi004', 'phi005', 'phi006']
uris = [f'{play_base}{pid}/phi0134.{pid}.perseus-lat2.xml' for pid in play_ids]

In [ ]:
TEI_NS = {'tei': 'http://www.tei-c.org/ns/1.0'}


def get_speaker_data(uri):
    """Return ({speaker: [line_text, ...]}, play_title) for one Perseus TEI play."""
    with urllib.request.urlopen(uri) as f:
        tree = etree.parse(f)
    root = tree.getroot()
    title = root.xpath('.//tei:title', namespaces=TEI_NS)[0].text
    blocks = defaultdict(list)
    for sp in root.xpath('.//tei:sp', namespaces=TEI_NS):
        speaker = sp.xpath('tei:speaker', namespaces=TEI_NS)[0].text
        for line in sp.xpath('tei:l', namespaces=TEI_NS):
            blocks[speaker].append(f'{line.text} {line.tail}'.strip())
    return blocks, title

In [ ]:
speaker_data = []
titles = []
for uri in uris:
    blocks, title = get_speaker_data(uri)
    speaker_data.append(blocks)
    titles.append(title)

print('Loaded plays:')
for t in titles:
    print(f'  - {t}')

## Dramatis personae per play (as encoded)

These are the raw speaker codes as they appear in the Perseus TEI—useful as a snapshot of the encoding, not as a clean dramatis personae.

In [ ]:
for title, blocks in zip(titles, speaker_data):
    print(f'{title}\n  {list(blocks.keys())}\n')

## Sample: first 10 lines of Demea (`De.`) in *Adelphoe*

In [ ]:
adelphoe = speaker_data[5]
pprint(adelphoe['De.'][:10])

## Line counts per speaker per play

Note: a 'line' here is one `<l>` element under an `<sp>`, which can include partial verses split across speakers (interrupted lines).

In [ ]:
for title, blocks in zip(titles, speaker_data):
    print(f'{title}')
    for speaker, lines in blocks.items():
        print(f'  {speaker}: {len(lines)}')
    print()

## Approximate word counts per speaker per play

(Character counts of the joined line text, not whitespace-tokenized words—rough but consistent.)

In [ ]:
for title, blocks in zip(titles, speaker_data):
    print(f'{title}')
    for speaker, lines in blocks.items():
        chars = sum(len(line) for line in lines)
        print(f'  {speaker}: {chars}')
    print()